In [1]:
import pandas as pd
import scipy as sp
import numpy as np
from lightfm import cross_validation
from lightfm.evaluation import precision_at_k
from lightfm.evaluation import recall_at_k
from lightfm.evaluation import auc_score
from lightfm import LightFM
import matplotlib.pyplot as plt
import pickle
import os

In [13]:
dhs = pd.read_table('dat_bin_FDR01_hg38.txt.gz', header=None).T
dhs.shape

(733, 3591898)

In [14]:
data_coo = sp.sparse.coo_matrix(dhs.values)

**Example with trained model**

In [15]:
pickle_in = open('/home/ctrader/Hold/DHS/DHS_model_e15_d16_reg0001.pickle', 'rb')
model_e15_d16_reg0001=pickle.load(pickle_in)

pickle_in = open('/home/ctrader/Hold/DHS/DHS_data_e15_d16_reg0001.pickle', 'rb')
data_e15_d16_reg0001=pickle.load(pickle_in)

In [16]:
n_biosamples, n_DHSs = data_e15_d16_reg0001['test'].shape

In [17]:
scores = model_e15_d16_reg0001.predict(10, np.arange(n_DHSs))

In [53]:
ordered = np.argsort(-scores)

In [54]:
topk = ordered[:100]

In [20]:
csr_data = data_e15_d16_reg0001['train'].tocsr()

In [21]:
sum(csr_data[10, ordered].toarray()[0][:100])

77.0

**Minimal example**

In [24]:
example_data = np.random.randint(2, size=100)
example_data = example_data.reshape(10, 10)
print(example_data)

[[0 1 0 1 1 1 1 0 0 1]
 [0 1 1 0 0 0 0 0 1 1]
 [1 0 1 0 1 1 1 1 0 0]
 [0 1 0 1 0 1 1 1 1 0]
 [0 0 0 0 0 1 0 0 1 0]
 [0 0 1 0 0 0 0 0 1 1]
 [0 0 1 0 1 1 1 1 0 0]
 [0 0 0 0 1 1 1 0 1 1]
 [1 0 1 0 0 0 1 0 0 0]
 [0 1 1 1 0 1 1 0 0 0]]


In [39]:
example_coo = sp.sparse.coo_matrix(example_data)
(example_train, example_test) = cross_validation.random_train_test_split(example_coo, test_percentage=0.2)

In [47]:
print(example_train.toarray())

[[0. 0. 0. 1. 1. 1. 1. 0. 0. 1.]
 [0. 1. 1. 0. 0. 0. 0. 0. 1. 1.]
 [1. 0. 0. 0. 1. 0. 1. 1. 0. 0.]
 [0. 1. 0. 1. 0. 1. 1. 0. 1. 0.]
 [0. 0. 0. 0. 0. 1. 0. 0. 1. 0.]
 [0. 0. 1. 0. 0. 0. 0. 0. 1. 1.]
 [0. 0. 1. 0. 1. 1. 1. 1. 0. 0.]
 [0. 0. 0. 0. 0. 0. 1. 0. 1. 1.]
 [0. 0. 0. 0. 0. 0. 1. 0. 0. 0.]
 [0. 1. 1. 1. 0. 0. 1. 0. 0. 0.]]


In [48]:
print(example_test.toarray())

[[0 1 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0]
 [0 0 1 0 0 1 0 0 0 0]
 [0 0 0 0 0 0 0 1 0 0]
 [0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 1 1 0 0 0 0]
 [1 0 1 0 0 0 0 0 0 0]
 [0 0 0 0 0 1 0 0 0 0]]


In [42]:
example_model = LightFM(no_components=1, loss='warp')
example_model.fit(example_train, epochs=1)

In [45]:
n_rows, n_cols = example_test.shape
print(n_rows, n_cols)

10 10


In [60]:
example_scores = example_model.predict(0, np.arange(n_cols))
print(example_scores)

[-0.39510924 -0.33510923 -0.23583423 -0.20696434 -0.20903921 -0.06512702
  0.09310129 -0.4428017  -0.12040272 -0.40209568]


In [72]:
example_ordered = np.argsort(-example_scores)
print(example_ordered)

[6 5 8 3 4 2 1 0 9 7]


In [62]:
example_topk = example_ordered[:10]

In [79]:
k = 6

In [80]:
example_train_csr = example_train.tocsr()
sum(example_train_csr[0, example_ordered].toarray()[0][:k])

4.0

In [81]:
example_test_csr = example_test.tocsr()
sum(example_test_csr[0, example_ordered].toarray()[0][:k])

0